# song-search -> Geode port build (Windows x86_64)

Cross-compiles `matcool.song-search` for **Geometry Dash 2.2081 (x86_64 / Win64)** on a
Linux kernel, per the Geode docs ("Required C++ Tools" -> Linux) plus matcool's own
`linux-win-cross` gist.

Heavy downloads (xwin splat, Geode SDK, bindings, toolchain) live under `/tmp/work`,
so the kernel output stays small: the mod source, the `.geode`, and `BUILD-LOG.txt`.

1. Install clang / cmake / git / ninja / lld
2. `xwin` splat the MSVC CRT + Windows SDK
3. `matcool/clang-msvc-sdk` CMake toolchain (clang-cl, `x86_64-pc-windows-msvc`)
4. Geode CLI + Geode SDK + 2.2081 bindings
5. Write the mod source, run the pure-C++ logic test natively
6. Configure + build the `.geode`
7. Verify the built DLL architecture and the `mod.json`

CPU only, no GPU/TPU.

In [ ]:
%%bash

set -uo pipefail
mkdir -p /tmp/work
LOG=/kaggle/working/BUILD-LOG.txt
: > "$LOG"
log(){ echo "== $* ==" | tee -a "$LOG"; }

# --- deps as the Geode docs "Required C++ Tools -> Linux" describe ---
log "1. deps: clang clang-tools lld llvm cmake git"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq >/dev/null 2>&1 || echo "apt update failed" >> "$LOG"
apt-get install -y -qq --no-install-recommends \
    git cmake ninja-build clang clang-tools lld llvm libssl3 \
    unzip xz-utils zip ca-certificates curl wget gpg >/dev/null 2>&1 \
  || echo "apt install partially failed" >> "$LOG"

# The docs note that some distros ship clang cmake that is too old: Geode
# needs Clang 19+ (21+ recommended) and CMake 3.29+, so pull both from the
# upstream repos when the default is too old.
. /etc/os-release
CODENAME=${VERSION_CODENAME:-noble}

CLANG_VER=$(clang --version | grep -oP 'version [0-9]+' | head -1 | grep -oP '[0-9]+')
if [ -z "$CLANG_VER" ] || [ "$CLANG_VER" -lt 19 ]; then
  log "clang ${CLANG_VER:-none} < 19 -> apt.llvm.org llvm-toolchain-$CODENAME-19"
  wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/llvm.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/llvm.gpg] https://apt.llvm.org/${CODENAME}/ llvm-toolchain-${CODENAME}-19 main" \
    > /etc/apt/sources.list.d/llvm19.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq --no-install-recommends clang-19 lld-19 llvm-19 \
    >/dev/null 2>&1 || echo "apt.llvm.org clang-19 failed" >> "$LOG"
  export PATH=/usr/lib/llvm-19/bin:$PATH
  ln -sf /usr/lib/llvm-19/bin/* /usr/local/bin/ 2>/dev/null || true
  hash -r
fi

CMAKE_VER=$(cmake --version | head -1 | grep -oP '[0-9]+\.[0-9]+' | head -1)
if [ -n "$CMAKE_VER" ] && awk "BEGIN{exit !($CMAKE_VER < 3.29)}"; then
  log "cmake $CMAKE_VER < 3.29 -> apt.kitware.com"
  wget -qO- https://apt.kitware.com/keys/kitware-archive-latest.asc 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/kitware.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/kitware.gpg] https://apt.kitware.com/ubuntu ${CODENAME} main" \
    > /etc/apt/sources.list.d/kitware.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq cmake >/dev/null 2>&1 || true
  hash -r
fi

{
  echo "cmake : $(cmake --version | head -1)"
  echo "git   : $(git --version)"
  echo "ninja : $(ninja --version)"
  echo "clang : $(clang --version | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

# Debian/Ubuntu never ship the clang-cl binary, and upstream dropped the
# x86_64 linux tarballs after 18.x. clang selects the CL driver purely from
# argv[0], so a link to clang named "clang-cl" is equivalent.
CLANG_DIR=$(dirname "$(command -v clang)")
if ! command -v clang-cl >/dev/null; then
  log "clang-cl absent -> linking $CLANG_DIR/clang as clang-cl"
  ln -sf "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl" 2>/dev/null \
    || cp -f "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl"
  ln -sf "$CLANG_DIR/clang" /usr/local/bin/clang-cl 2>/dev/null || true
  hash -r
fi

{
  echo "clang : $(clang --version | head -1)"
  echo "clang-cl: $(clang-cl --version 2>&1 | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

command -v clang-cl >/dev/null || { echo "FATAL clang-cl missing" | tee -a "$LOG"; exit 2; }
command -v lld-link >/dev/null || { echo "FATAL lld-link missing" | tee -a "$LOG"; exit 2; }
log "DEPS DONE"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "2. xwin splat (MSVC CRT + Windows SDK)"

XWIN_VER=0.10.0
if [ ! -x /tmp/work/xwin/xwin ]; then
  curl -sSL -o /tmp/work/xwin.tar.gz \
    "https://github.com/Jake-Shadle/xwin/releases/download/${XWIN_VER}/xwin-${XWIN_VER}-x86_64-unknown-linux-musl.tar.gz" \
    || { echo "xwin download failed" | tee -a "$LOG"; exit 3; }
  mkdir -p /tmp/work/xwin
  tar -xzf /tmp/work/xwin.tar.gz -C /tmp/work/xwin --strip-components=1
  chmod +x /tmp/work/xwin/xwin
fi
export PATH=/tmp/work/xwin:$PATH
xwin --version | tee -a "$LOG"

# xwin's cache skips already-downloaded payloads, so retrying accumulates
# progress through a flaky connection. Default --arch is x86_64 and the
# default --variant is desktop; debug libs are excluded (Release build).
export XWIN_ACCEPT_LICENSE=1
DONE=0
for attempt in $(seq 1 60); do
  echo "--- xwin splat attempt $attempt/60 ---" >> "$LOG"
  # NB: --accept-license/--arch/--variant/--timeout/--cache-dir are GLOBAL
  # (before the subcommand); --output belongs to the splat subcommand.
  xwin --accept-license --arch x86_64 --variant desktop \
       --timeout 300 --cache-dir /tmp/work/xwin-cache \
       splat --include-debug-libs --output /tmp/work/splat >>"$LOG" 2>&1
  RC=$?
  if [ $RC -eq 0 ]; then
    DONE=1
    log "xwin splat OK on attempt $attempt"
    break
  fi
  echo "attempt $attempt rc=$RC" >> "$LOG"
  sleep 3
done
echo "xwin done=$DONE" | tee -a "$LOG"

echo "--- splat layout ---" | tee -a "$LOG"
du -sh /tmp/work/splat 2>>"$LOG" || true
for d in crt/include crt/lib/x86_64 \
         sdk/include/ucrt sdk/include/um sdk/include/shared sdk/include/winrt \
         sdk/lib/ucrt/x86_64 sdk/lib/um/x86_64; do
  if [ -d "/tmp/work/splat/$d" ]; then
    echo "  OK   $d" | tee -a "$LOG"
  else
    echo "  MISS $d" | tee -a "$LOG"
  fi
done
ls /tmp/work/splat/sdk/include/um/Windows.h >/dev/null 2>&1 \
  && echo "  Windows.h present" | tee -a "$LOG" \
  || echo "  Windows.h MISSING" | tee -a "$LOG"
[ "$DONE" = "1" ] || exit 4

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "3. toolchain + Geode CLI + Geode SDK + bindings"
mkdir -p /tmp/work && cd /tmp/work || exit 5
export PATH=/usr/local/bin:$PATH

if [ ! -d toolchain/.git ]; then
  git clone --depth 1 https://github.com/matcool/clang-msvc-sdk.git toolchain >>"$LOG" 2>&1 \
    || { echo "toolchain clone failed" | tee -a "$LOG"; exit 6; }
fi
ls toolchain | tee -a "$LOG"

GEODE_CLI_VER=v3.9.0
if [ ! -x /tmp/work/geode-cli/geode ]; then
  curl -sSL -o /tmp/work/geode.zip \
    "https://github.com/geode-sdk/cli/releases/download/${GEODE_CLI_VER}/geode-cli-${GEODE_CLI_VER}-linux.zip" \
    || { echo "geode cli download failed" | tee -a "$LOG"; exit 7; }
  rm -rf /tmp/work/geode-cli
  unzip -q -o /tmp/work/geode.zip -d /tmp/work/geode-cli
  chmod +x /tmp/work/geode-cli/geode
fi
export PATH=/tmp/work/geode-cli:$PATH
geode --version | tee -a "$LOG"

# `geode project check` (run by the SDK's CMake) needs a current CLI profile.
# Configure a throwaway one pointing at a scratch dir - we only cross-build
# here, never run the game. It also wants the usual Geode install layout
# present so it can read the list of installed mods.
mkdir -p /tmp/work/gd-scratch/geode/mods
mkdir -p /tmp/work/gd-scratch/geode/config
mkdir -p /tmp/work/gd-scratch/geode/logs
mkdir -p /tmp/work/gd-scratch/geode/unzipped
if ! geode profile list 2>/dev/null | grep -q kaggle; then
  geode profile add --name kaggle /tmp/work/gd-scratch >>"$LOG" 2>&1 || true
fi
geode profile switch kaggle >>"$LOG" 2>&1 || true
echo "profiles:" | tee -a "$LOG"
geode profile list 2>&1 | tee -a "$LOG"

if [ ! -d sdk/.git ]; then
  git clone --depth 1 --branch main https://github.com/geode-sdk/sdk.git sdk >>"$LOG" 2>&1 \
    || { echo "geode sdk clone failed" | tee -a "$LOG"; exit 8; }
fi
geode sdk set-path "/tmp/work/sdk" >>"$LOG" 2>&1 || true
geode sdk version | tee -a "$LOG"
ls sdk/loader/include/link/win64 | tee -a "$LOG"

# The SDK links the Geode loader import library from <sdk>/bin/<version>/.
# `geode sdk install-binaries` fetches it; do it manually if the CLI cannot.
SDK_VER=$(cat sdk/VERSION 2>/dev/null | tr -d '[:space:]')
echo "sdk version: $SDK_VER" | tee -a "$LOG"
if [ -n "$SDK_VER" ] && [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  geode sdk install-binaries >>"$LOG" 2>&1 || true
fi
if [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  log "install-binaries did not provide Geode.lib -> fetching loader zip directly"
  for attempt in $(seq 1 20); do
    curl -sSL -o /tmp/work/geode-win.zip \
      "https://github.com/geode-sdk/geode/releases/download/v${SDK_VER}/geode-v${SDK_VER}-win.zip" && break
    echo "loader zip attempt $attempt failed" >> "$LOG"
    sleep 5
  done
  rm -rf /tmp/work/geode-win && mkdir -p /tmp/work/geode-win
  unzip -q -o /tmp/work/geode-win.zip -d /tmp/work/geode-win || true
  mkdir -p "sdk/bin/${SDK_VER}"
  FOUND=$(find /tmp/work/geode-win -name 'Geode.lib' | head -1)
  if [ -n "$FOUND" ]; then
    cp -f "$FOUND" "sdk/bin/${SDK_VER}/Geode.lib"
    log "installed Geode.lib from $FOUND"
  else
    echo "WARNING: Geode.lib not found in loader zip" | tee -a "$LOG"
    find /tmp/work/geode-win -maxdepth 3 -type f | head -30 | tee -a "$LOG"
  fi
fi
ls -la "sdk/bin/${SDK_VER}/" 2>&1 | tee -a "$LOG"

# Geode generates binding headers at configure time from the bindings repo;
# pre-seed a clone so configure does not fetch it itself
if [ ! -d bindings/.git ]; then
  git clone --depth 1 https://github.com/geode-sdk/bindings.git bindings >>"$LOG" 2>&1 \
    || { echo "bindings clone failed" | tee -a "$LOG"; exit 9; }
fi
ls bindings/bindings/2.2081 | tee -a "$LOG"
log "TOOLCHAIN DONE"

In [ ]:
import base64, json, pathlib
FILES = {
 "mod.json": "ewoJImdlb2RlIjogIjUuMTEuMSIsCgkiZ2QiOiB7CgkJIndpbiI6ICIqIiwKCQkiYW5kcm9pZCI6ICIqIiwKCQkibWFjIjogIioiLAoJCSJpb3MiOiAiKiIKCX0sCgkiaWQiOiAibWF0Y29vbC5zb25nLXNlYXJjaCIsCgkibmFtZSI6ICJTb25nIFNlYXJjaCIsCgkidmVyc2lvbiI6ICJ2MC4yLjAiLAoJImRldmVsb3BlcnMiOiBbCgkJIm5vdGxvdXN5Ym9vayIsCgkJIm1hdGNvb2wiCgldLAoJImRlc2NyaXB0aW9uIjogIlNlYXJjaCBhbmQgZmlsdGVyIGRvd25sb2FkZWQgc29uZ3Mgd2l0aCBmdXp6eSBtYXRjaGluZywgcGx1cyBzb3J0YWJsZSBmaWx0ZXIgYnV0dG9ucyBhbmQgSnVrZWJveCBOT05HIHN1cHBvcnQuIiwKCSJ0YWdzIjogWwoJCSJtdXNpYyIsCgkJImludGVyZmFjZSIsCgkJImVuaGFuY2VtZW50IgoJXSwKCSJzZXR0aW5ncyI6IHsKCQkiZ2VuZXJhbCI6IHsKCQkJInR5cGUiOiAidGl0bGUiLAoJCQkibmFtZSI6ICJHZW5lcmFsIgoJCX0sCgkJImRpc2FibGUtbW9kIjogewoJCQkidHlwZSI6ICJib29sIiwKCQkJIm5hbWUiOiAiRGlzYWJsZSBTb25nIFNlYXJjaCIsCgkJCSJkZXNjcmlwdGlvbiI6ICJUdXJuIHRoZSBzZWFyY2ggZGlhbG9nIG9mZiBlbnRpcmVseSBhbmQgcmVzdG9yZSB0aGUgZGVmYXVsdCBzb25nIGJyb3dzZXIuIiwKCQkJImRlZmF1bHQiOiBmYWxzZQoJCX0sCgkJInNob3ctZmlsdGVycyI6IHsKCQkJInR5cGUiOiAiYm9vbCIsCgkJCSJuYW1lIjogIlNob3cgZmlsdGVyIGJ1dHRvbnMiLAoJCQkiZGVzY3JpcHRpb24iOiAiU2hvdyB0aGUgc29ydC9maWx0ZXIgdG9nZ2xlIGJ1dHRvbnMgaW4gdGhlIGNvcm5lciBvZiB0aGUgc2VhcmNoIGRpYWxvZy4iLAoJCQkiZGVmYXVsdCI6IHRydWUKCQl9LAoJCSJqdWtlYm94IjogewoJCQkidHlwZSI6ICJ0aXRsZSIsCgkJCSJuYW1lIjogIkp1a2Vib3ggTk9ORyBzdXBwb3J0IgoJCX0sCgkJIm5vbmctc3ltYm9scyI6IHsKCQkJInR5cGUiOiAiYm9vbCIsCgkJCSJuYW1lIjogIlNob3cgTk9ORyBzeW1ib2xzIiwKCQkJImRlc2NyaXB0aW9uIjogIlNob3cgYSBkaXNjIG9uIHNvbmdzIHRoYXQgaGF2ZSBhIE5PTkcgc2V0IGluIEp1a2Vib3guIFJlcXVpcmVzIEp1a2Vib3ggdG8gYmUgaW5zdGFsbGVkOyBkb2VzIG5vdGhpbmcgd2l0aG91dCBpdC4iLAoJCQkiZGVmYXVsdCI6IHRydWUKCQl9LAoJCSJ1c2Utbm9uZy1tZXRhZGF0YSI6IHsKCQkJInR5cGUiOiAiYm9vbCIsCgkJCSJuYW1lIjogIlVzZSBOT05HIG5hbWUgYW5kIGFydGlzdCIsCgkJCSJkZXNjcmlwdGlvbiI6ICJEaXNwbGF5IHRoZSBOT05HJ3MgbmFtZSBhbmQgYXJ0aXN0IGluIHJlc3VsdHMgd2hlbiBhdmFpbGFibGUgaW5zdGVhZCBvZiB0aGUgb3JpZ2luYWwgTmV3Z3JvdW5kcyBzb25nLiIsCgkJCSJkZWZhdWx0IjogdHJ1ZQoJCX0KCX0sCgkicmVzb3VyY2VzIjogewoJCSJzcHJpdGVzIjogWwoJCQkicmVzb3VyY2VzLyoucG5nIgoJCV0KCX0sCgkibGlua3MiOiB7CgkJInNvdXJjZSI6ICJodHRwczovL2dpdGh1Yi5jb20vbWF0Y29vbC9zbWFsbC1nZC1tb2RzIgoJfSwKCSJpc3N1ZXMiOiB7CgkJImluZm8iOiAiUmVwb3J0IGlzc3VlcyBvbiB0aGUgR2l0SHViIHJlcG9zaXRvcnkgb2YgdGhlIG9yaWdpbmFsIG1vZCwgb3IgaW4gdGhlIFNvbmcgU2VhcmNoIHRocmVhZC4iLAoJCSJ1cmwiOiAiaHR0cHM6Ly9naXRodWIuY29tL21hdGNvb2wvc21hbGwtZ2QtbW9kcy9pc3N1ZXMiCgl9Cn0K",
 "CMakeLists.txt": "Y21ha2VfbWluaW11bV9yZXF1aXJlZChWRVJTSU9OIDMuMjEpCnNldChDTUFLRV9DWFhfU1RBTkRBUkQgMjMpCnNldChDTUFLRV9DWFhfU1RBTkRBUkRfUkVRVUlSRUQgT04pCnNldChDTUFLRV9DWFhfVklTSUJJTElUWV9QUkVTRVQgaGlkZGVuKQoKcHJvamVjdChzb25nLXNlYXJjaCBWRVJTSU9OIDEuMC4wKQoKIyBBZGQgYWxsIHNvdXJjZSBmaWxlcyBpbnNpZGUgc3JjIChyZWN1cnNpdmVseSkKZmlsZShHTE9CX1JFQ1VSU0UgU09VUkNFUyBDT05GSUdVUkVfREVQRU5EUyBzcmMvKi5jcHApCgojIFNldCB1cCB0aGUgbW9kIGJpbmFyeQphZGRfbGlicmFyeSgke1BST0pFQ1RfTkFNRX0gU0hBUkVEICR7U09VUkNFU30pCgppZiAoTk9UIERFRklORUQgRU5We0dFT0RFX1NES30pCiAgICBtZXNzYWdlKEZBVEFMX0VSUk9SICJVbmFibGUgdG8gZmluZCBHZW9kZSBTREshIFBsZWFzZSBkZWZpbmUgR0VPREVfU0RLIGVudmlyb25tZW50IHZhcmlhYmxlIHRvIHBvaW50IHRvIEdlb2RlIikKZWxzZSgpCiAgICBtZXNzYWdlKFNUQVRVUyAiRm91bmQgR2VvZGU6ICRFTlZ7R0VPREVfU0RLfSIpCmVuZGlmKCkKCmFkZF9zdWJkaXJlY3RvcnkoJEVOVntHRU9ERV9TREt9ICR7Q01BS0VfQ1VSUkVOVF9CSU5BUllfRElSfS9nZW9kZSkKCiMgU2V0IHVwIGRlcGVuZGVuY2llcywgcmVzb3VyY2VzLCBhbmQgbGluayBHZW9kZS4Kc2V0dXBfZ2VvZGVfbW9kKCR7UFJPSkVDVF9OQU1FfSkK",
 ".gitignore": "YnVpbGQvCi50b29scy8KKi5nZW9kZQoKIyBLYWdnbGUgYnVpbGQgc2NyYXRjaCBvdXRwdXQgKGxvZ3MgLyBmZXRjaGVkIHBhY2thZ2VzKQoua2FnZ2xlLyovb3V0Ki8KLmthZ2dsZS8qL28qLwoua2FnZ2xlLyovcGtnLwoua2FnZ2xlLyovcHVsbGVkKi8KCiMgS2lsby1pbnN0YWxsZWQgc2tpbGwgY29weSAoc291cmNlIG9mIHRydXRoIGlzIC5rYWdnbGUvc2tpbGxzLykKLmtpbG9jb2RlLwo=",
 "src/main.cpp": "...",
 "src/fts_fuzzy_match.h": "Ly8gTElDRU5TRQovLwovLyAgIFRoaXMgc29mdHdhcmUgaXMgZHVhbC1saWNlbnNlZCB0byB0aGUgcHVibGljIGRvbWFpbiBhbmQgdW5kZXIgdGhlIGZvbGxvd2luZwovLyAgIGxpY2Vuc2U6IHlvdSBhcmUgZ3JhbnRlZCBhIHBlcnBldHVhbCwgaXJyZXZvY2FibGUgbGljZW5zZSB0byBjb3B5LCBtb2RpZnksCi8vICAgcHVibGlzaCwgYW5kIGRpc3RyaWJ1dGUgdGhpcyBmaWxlIGFzIHlvdSBzZWUgZml0LgovLwovLyBWRVJTSU9OCi8vICAgMC4yLjAgICgyMDE3LTAyLTE4KSAgU2NvcmVkIG1hdGNoZXMgcGVyZm9ybSBleGhhdXN0aXZlIHNlYXJjaCBmb3IgYmVzdCBzY29yZQovLyAgIDAuMS4wICAoMjAxNi0wMy0yOCkgIEluaXRpYWwgcmVsZWFzZQovLwovLyBBVVRIT1IKLy8gICBGb3JyZXN0IFNtaXRoCi8vCi8vIE5PVEVTCi8vICAgQ29tcGlsaW5nCi8vICAgICBZb3UgTVVTVCBhZGQgJyNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OJyBiZWZvcmUgaW5jbHVkaW5nIHRoaXMgaGVhZGVyIGluIE9ORSBzb3VyY2UgZmlsZSB0byBjcmVhdGUgaW1wbGVtZW50YXRpb24uCi8vCi8vICAgZnV6enlfbWF0Y2hfc2ltcGxlKC4uLikKLy8gICAgIFJldHVybnMgdHJ1ZSBpZiBlYWNoIGNoYXJhY3RlciBpbiBwYXR0ZXJuIGlzIGZvdW5kIHNlcXVlbnRpYWxseSB3aXRoaW4gc3RyCi8vCi8vICAgZnV6enlfbWF0Y2goLi4uKQovLyAgICAgUmV0dXJucyB0cnVlIGlmIHBhdHRlcm4gaXMgZm91bmQgQU5EIGNhbGN1bGF0ZXMgYSBzY29yZS4KLy8gICAgIFBlcmZvcm1zIGV4aGF1c3RpdmUgc2VhcmNoIHZpYSByZWN1cnNpb24gdG8gZmluZCBhbGwgcG9zc2libGUgbWF0Y2hlcyBhbmQgbWF0Y2ggd2l0aCBoaWdoZXN0IHNjb3JlLgovLyAgICAgU2NvcmVzIHZhbHVlcyBoYXZlIG5vIGludHJpbnNpYyBtZWFuaW5nLiBQb3NzaWJsZSBzY29yZSByYW5nZSBpcyBub3Qgbm9ybWFsaXplZCBhbmQgdmFyaWVzIHdpdGggcGF0dGVybi4KLy8gICAgIFJlY3Vyc2lvbiBpcyBsaW1pdGVkIGludGVybmFsbHkgKGRlZmF1bHQ9MTApIHRvIHByZXZlbnQgZGVnZW5lcmF0ZSBjYXNlcyAocGF0dGVybj0iYWFhYWFhIiBzdHI9ImFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYSIpCi8vICAgICBVc2VzIHVpbnQ4X3QgZm9yIG1hdGNoIGluZGljZXMuIFRoZXJlZm9yZSBwYXR0ZXJucyBhcmUgbGltaXRlZCB0byAyNTYgY2hhcmFjdGVycy4KLy8gICAgIFNjb3JlIHN5c3RlbSBzaG91bGQgYmUgdHVuZWQgZm9yIFlPVVIgdXNlIGNhc2UuIFdvcmRzLCBzZW50ZW5jZXMsIGZpbGUgbmFtZXMsIG9yIG1ldGhvZCBuYW1lcyBhbGwgcHJlZmVyIGRpZmZlcmVudCB0dW5pbmcuCgojaWZuZGVmIEZUU19GVVpaWV9NQVRDSF9ICiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0gKCiNpbmNsdWRlIDxjc3RkaW50PiAvLyB1aW50OF90CiNpbmNsdWRlIDxjdHlwZS5oPiAvLyA6OnRvbG93ZXIsIDo6dG91cHBlcgojaW5jbHVkZSA8Y3N0cmluZz4gLy8gbWVtY3B5CgojaW5jbHVkZSA8Y3N0ZGlvPgoKLy8gUHVibGljIGludGVyZmFjZQpuYW1lc3BhY2UgZnRzIHsKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cik7CiAgICBzdGF0aWMgYm9vbCBmdXp6eV9tYXRjaChjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0ciwgaW50ICYgb3V0U2NvcmUpOwogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlLCB1aW50OF90ICogbWF0Y2hlcywgaW50IG1heE1hdGNoZXMpOwp9CgojaWZkZWYgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCm5hbWVzcGFjZSBmdHMgewoKICAgIC8vIEZvcndhcmQgZGVjbGFyYXRpb25zIGZvciAicHJpdmF0ZSIgaW1wbGVtZW50YXRpb24KICAgIG5hbWVzcGFjZSBmdXp6eV9pbnRlcm5hbCB7CiAgICAgICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2hfcmVjdXJzaXZlKGNvbnN0IGNoYXIgKiBwYXR0ZXJuLCBjb25zdCBjaGFyICogc3RyLCBpbnQgJiBvdXRTY29yZSwgY29uc3QgY2hhciAqIHN0ckJlZ2luLAogICAgICAgICAgICB1aW50OF90IGNvbnN0ICogc3JjTWF0Y2hlcywgIHVpbnQ4X3QgKiBuZXdNYXRjaGVzLCAgaW50IG1heE1hdGNoZXMsIGludCBuZXh0TWF0Y2gsCiAgICAgICAgICAgIGludCAmIHJlY3Vyc2lvbkNvdW50LCBpbnQgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFB1YmxpYyBpbnRlcmZhY2UKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cikgewogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgIHsKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpCiAgICAgICAgICAgICAgICArK3BhdHRlcm47CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgcmV0dXJuICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CiAgICB9CgogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlKSB7CgogICAgICAgIHVpbnQ4X3QgbWF0Y2hlc1syNTZdOwogICAgICAgIHJldHVybiBmdXp6eV9tYXRjaChwYXR0ZXJuLCBzdHIsIG91dFNjb3JlLCBtYXRjaGVzLCBzaXplb2YobWF0Y2hlcykpOwogICAgfQoKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoKGNoYXIgY29uc3QgKiBwYXR0ZXJuLCBjaGFyIGNvbnN0ICogc3RyLCBpbnQgJiBvdXRTY29yZSwgdWludDhfdCAqIG1hdGNoZXMsIGludCBtYXhNYXRjaGVzKSB7CiAgICAgICAgaW50IHJlY3Vyc2lvbkNvdW50ID0gMDsKICAgICAgICBpbnQgcmVjdXJzaW9uTGltaXQgPSAxMDsKCiAgICAgICAgcmV0dXJuIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyLCBvdXRTY29yZSwgc3RyLCBudWxscHRyLCBtYXRjaGVzLCBtYXhNYXRjaGVzLCAwLCByZWN1cnNpb25Db3VudCwgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFByaXZhdGUgaW1wbGVtZW50YXRpb24KICAgIHN0YXRpYyBib29sIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUoY29uc3QgY2hhciAqIHBhdHRlcm4sIGNvbnN0IGNoYXIgKiBzdHIsIGludCAmIG91dFNjb3JlLAogICAgICAgIGNvbnN0IGNoYXIgKiBzdHJCZWdpbiwgdWludDhfdCBjb25zdCAqIHNyY01hdGNoZXMsIHVpbnQ4X3QgKiBtYXRjaGVzLCBpbnQgbWF4TWF0Y2hlcywKICAgICAgICBpbnQgbmV4dE1hdGNoLCBpbnQgJiByZWN1cnNpb25Db3VudCwgaW50IHJlY3Vyc2lvbkxpbWl0KQogICAgewogICAgICAgIC8vIENvdW50IHJlY3Vyc2lvbnMKICAgICAgICArK3JlY3Vyc2lvbkNvdW50OwogICAgICAgIGlmIChyZWN1cnNpb25Db3VudCA+PSByZWN1cnNpb25MaW1pdCkKICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAvLyBEZXRlY3QgZW5kIG9mIHN0cmluZ3MKICAgICAgICBpZiAoKnBhdHRlcm4gPT0gJ1wwJyB8fCAqc3RyID09ICdcMCcpCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKCiAgICAgICAgLy8gUmVjdXJzaW9uIHBhcmFtcwogICAgICAgIGJvb2wgcmVjdXJzaXZlTWF0Y2ggPSBmYWxzZTsKICAgICAgICB1aW50OF90IGJlc3RSZWN1cnNpdmVNYXRjaGVzWzI1Nl07CiAgICAgICAgaW50IGJlc3RSZWN1cnNpdmVTY29yZSA9IDA7CgogICAgICAgIC8vIExvb3AgdGhyb3VnaCBwYXR0ZXJuIGFuZCBzdHIgbG9va2luZyBmb3IgYSBtYXRjaAogICAgICAgIGJvb2wgZmlyc3RfbWF0Y2ggPSB0cnVlOwogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgewoKICAgICAgICAgICAgLy8gRm91bmQgbWF0Y2gKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpIHsKCiAgICAgICAgICAgICAgICAvLyBTdXBwbGllZCBtYXRjaGVzIGJ1ZmZlciB3YXMgdG9vIHNob3J0CiAgICAgICAgICAgICAgICBpZiAobmV4dE1hdGNoID49IG1heE1hdGNoZXMpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAgICAgICAgIC8vICJDb3B5LW9uLVdyaXRlIiBzcmNNYXRjaGVzIGludG8gbWF0Y2hlcwogICAgICAgICAgICAgICAgaWYgKGZpcnN0X21hdGNoICYmIHNyY01hdGNoZXMpIHsKICAgICAgICAgICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgc3JjTWF0Y2hlcywgbmV4dE1hdGNoKTsKICAgICAgICAgICAgICAgICAgICBmaXJzdF9tYXRjaCA9IGZhbHNlOwogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgICAgIC8vIFJlY3Vyc2l2ZSBjYWxsIHRoYXQgInNraXBzIiB0aGlzIG1hdGNoCiAgICAgICAgICAgICAgICB1aW50OF90IHJlY3Vyc2l2ZU1hdGNoZXNbMjU2XTsKICAgICAgICAgICAgICAgIGludCByZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgICAgIGlmIChmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyICsgMSwgcmVjdXJzaXZlU2NvcmUsIHN0ckJlZ2luLCBtYXRjaGVzLCByZWN1cnNpdmVNYXRjaGVzLCBzaXplb2YocmVjdXJzaXZlTWF0Y2hlcyksIG5leHRNYXRjaCwgcmVjdXJzaW9uQ291bnQsIHJlY3Vyc2lvbkxpbWl0KSkgewoKICAgICAgICAgICAgICAgICAgICAvLyBQaWNrIGJlc3QgcmVjdXJzaXZlIHNjb3JlCiAgICAgICAgICAgICAgICAgICAgaWYgKCFyZWN1cnNpdmVNYXRjaCB8fCByZWN1cnNpdmVTY29yZSA+IGJlc3RSZWN1cnNpdmVTY29yZSkgewogICAgICAgICAgICAgICAgICAgICAgICBtZW1jcHkoYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIHJlY3Vyc2l2ZU1hdGNoZXMsIDI1Nik7CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RSZWN1cnNpdmVTY29yZSA9IHJlY3Vyc2l2ZVNjb3JlOwogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICByZWN1cnNpdmVNYXRjaCA9IHRydWU7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQWR2YW5jZQogICAgICAgICAgICAgICAgbWF0Y2hlc1tuZXh0TWF0Y2grK10gPSAodWludDhfdCkoc3RyIC0gc3RyQmVnaW4pOwogICAgICAgICAgICAgICAgKytwYXR0ZXJuOwogICAgICAgICAgICB9CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgLy8gRGV0ZXJtaW5lIGlmIGZ1bGwgcGF0dGVybiB3YXMgbWF0Y2hlZAogICAgICAgIGJvb2wgbWF0Y2hlZCA9ICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CgogICAgICAgIC8vIENhbGN1bGF0ZSBzY29yZQogICAgICAgIGlmIChtYXRjaGVkKSB7CiAgICAgICAgICAgIGNvbnN0IGludCBzZXF1ZW50aWFsX2JvbnVzID0gMTU7ICAgICAgICAgICAgLy8gYm9udXMgZm9yIGFkamFjZW50IG1hdGNoZXMKICAgICAgICAgICAgY29uc3QgaW50IHNlcGFyYXRvcl9ib251cyA9IDMwOyAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBvY2N1cnMgYWZ0ZXIgYSBzZXBhcmF0b3IKICAgICAgICAgICAgY29uc3QgaW50IGNhbWVsX2JvbnVzID0gMzA7ICAgICAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBpcyB1cHBlcmNhc2UgYW5kIHByZXYgaXMgbG93ZXIKICAgICAgICAgICAgY29uc3QgaW50IGZpcnN0X2xldHRlcl9ib251cyA9IDE1OyAgICAgICAgICAvLyBib251cyBpZiB0aGUgZmlyc3QgbGV0dGVyIGlzIG1hdGNoZWQKCiAgICAgICAgICAgIGNvbnN0IGludCBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTU7ICAgICAgLy8gcGVuYWx0eSBhcHBsaWVkIGZvciBldmVyeSBsZXR0ZXIgaW4gc3RyIGJlZm9yZSB0aGUgZmlyc3QgbWF0Y2gKICAgICAgICAgICAgY29uc3QgaW50IG1heF9sZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTE1OyAvLyBtYXhpbXVtIHBlbmFsdHkgZm9yIGxlYWRpbmcgbGV0dGVycwogICAgICAgICAgICBjb25zdCBpbnQgdW5tYXRjaGVkX2xldHRlcl9wZW5hbHR5ID0gLTE7ICAgIC8vIHBlbmFsdHkgZm9yIGV2ZXJ5IGxldHRlciB0aGF0IGRvZXNuJ3QgbWF0dGVyCgogICAgICAgICAgICAvLyBJdGVyYXRlIHN0ciB0byBlbmQKICAgICAgICAgICAgd2hpbGUgKCpzdHIgIT0gJ1wwJykKICAgICAgICAgICAgICAgICsrc3RyOwoKICAgICAgICAgICAgLy8gSW5pdGlhbGl6ZSBzY29yZQogICAgICAgICAgICBvdXRTY29yZSA9IDEwMDsKCiAgICAgICAgICAgIC8vIEFwcGx5IGxlYWRpbmcgbGV0dGVyIHBlbmFsdHkKICAgICAgICAgICAgaW50IHBlbmFsdHkgPSBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ICogbWF0Y2hlc1swXTsKICAgICAgICAgICAgaWYgKHBlbmFsdHkgPCBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eSkKICAgICAgICAgICAgICAgIHBlbmFsdHkgPSBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eTsKICAgICAgICAgICAgb3V0U2NvcmUgKz0gcGVuYWx0eTsKCiAgICAgICAgICAgIC8vIEFwcGx5IHVubWF0Y2hlZCBwZW5hbHR5CiAgICAgICAgICAgIGludCB1bm1hdGNoZWQgPSAoaW50KShzdHIgLSBzdHJCZWdpbikgLSBuZXh0TWF0Y2g7CiAgICAgICAgICAgIG91dFNjb3JlICs9IHVubWF0Y2hlZF9sZXR0ZXJfcGVuYWx0eSAqIHVubWF0Y2hlZDsKCiAgICAgICAgICAgIC8vIEFwcGx5IG9yZGVyaW5nIGJvbnVzZXMKICAgICAgICAgICAgZm9yIChpbnQgaSA9IDA7IGkgPCBuZXh0TWF0Y2g7ICsraSkgewogICAgICAgICAgICAgICAgdWludDhfdCBjdXJySWR4ID0gbWF0Y2hlc1tpXTsKCiAgICAgICAgICAgICAgICBpZiAoaSA+IDApIHsKICAgICAgICAgICAgICAgICAgICB1aW50OF90IHByZXZJZHggPSBtYXRjaGVzW2kgLSAxXTsKCiAgICAgICAgICAgICAgICAgICAgLy8gU2VxdWVudGlhbAogICAgICAgICAgICAgICAgICAgIGlmIChjdXJySWR4ID09IChwcmV2SWR4ICsgMSkpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcXVlbnRpYWxfYm9udXM7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQ2hlY2sgZm9yIGJvbnVzZXMgYmFzZWQgb24gbmVpZ2hib3IgY2hhcmFjdGVyIHZhbHVlCiAgICAgICAgICAgICAgICBpZiAoY3VycklkeCA+IDApIHsKICAgICAgICAgICAgICAgICAgICAvLyBDYW1lbCBjYXNlCiAgICAgICAgICAgICAgICAgICAgY2hhciBuZWlnaGJvciA9IHN0ckJlZ2luW2N1cnJJZHggLSAxXTsKICAgICAgICAgICAgICAgICAgICBjaGFyIGN1cnIgPSBzdHJCZWdpbltjdXJySWR4XTsKICAgICAgICAgICAgICAgICAgICBpZiAoOjppc2xvd2VyKG5laWdoYm9yKSAmJiA6OmlzdXBwZXIoY3VycikpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGNhbWVsX2JvbnVzOwoKICAgICAgICAgICAgICAgICAgICAvLyBTZXBhcmF0b3IKICAgICAgICAgICAgICAgICAgICBib29sIG5laWdoYm9yU2VwYXJhdG9yID0gbmVpZ2hib3IgPT0gJ18nIHx8IG5laWdoYm9yID09ICcgJzsKICAgICAgICAgICAgICAgICAgICBpZiAobmVpZ2hib3JTZXBhcmF0b3IpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcGFyYXRvcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVsc2UgewogICAgICAgICAgICAgICAgICAgIC8vIEZpcnN0IGxldHRlcgogICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGZpcnN0X2xldHRlcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgfQogICAgICAgIH0KCiAgICAgICAgLy8gUmV0dXJuIGJlc3QgcmVzdWx0CiAgICAgICAgaWYgKHJlY3Vyc2l2ZU1hdGNoICYmICghbWF0Y2hlZCB8fCBiZXN0UmVjdXJzaXZlU2NvcmUgPiBvdXRTY29yZSkpIHsKICAgICAgICAgICAgLy8gUmVjdXJzaXZlIHNjb3JlIGlzIGJldHRlciB0aGFuICJ0aGlzIgogICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIG1heE1hdGNoZXMpOwogICAgICAgICAgICBvdXRTY29yZSA9IGJlc3RSZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgcmV0dXJuIHRydWU7CiAgICAgICAgfQogICAgICAgIGVsc2UgaWYgKG1hdGNoZWQpIHsKICAgICAgICAgICAgLy8gInRoaXMiIHNjb3JlIGlzIGJldHRlciB0aGFuIHJlY3Vyc2l2ZQogICAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIC8vIG5vIG1hdGNoCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKICAgICAgICB9CiAgICB9Cn0gLy8gbmFtZXNwYWNlIGZ0cwoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9JTVBMRU1FTlRBVElPTgoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9ICg==",
 "tests/logic_test.cpp": "Ly8gU3RhbmRhbG9uZSBsb2dpYyB0ZXN0IGZvciB0aGUgc29uZy1zZWFyY2ggcG9ydC4KLy8gTWlycm9ycyB0aGUgZnV6enkgc2VhcmNoICsgc29ydCArIHNjcm9sbC13aW5kb3cgbG9naWMgaW4gc3JjL21haW4uY3BwCi8vIHdpdGhvdXQgR2VvZGUgaGVhZGVycywgc28gaXQgY2FuIHJ1biBvbiBwbGFpbiBHQ0MuCiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCiNpbmNsdWRlICJmdHNfZnV6enlfbWF0Y2guaCIKCiNpbmNsdWRlIDxhbGdvcml0aG0+CiNpbmNsdWRlIDxjYXNzZXJ0PgojaW5jbHVkZSA8Y21hdGg+CiNpbmNsdWRlIDxpb3N0cmVhbT4KI2luY2x1ZGUgPHNzdHJlYW0+CiNpbmNsdWRlIDxzdHJpbmc+CiNpbmNsdWRlIDx2ZWN0b3I+CgpzdHJ1Y3QgRmFrZVNvbmcgeyBpbnQgaWQ7IHN0ZDo6c3RyaW5nIG5hbWU7IHN0ZDo6c3RyaW5nIGFydGlzdDsgfTsKCnN0YXRpYyBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8aW50LCBGYWtlU29uZz4+IGRvX3NlYXJjaCgKICAgIGNvbnN0IHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxzdGQ6OnN0cmluZywgRmFrZVNvbmc+PiYgZG93bmxvYWRlZCwKICAgIGNvbnN0IHN0ZDo6c3RyaW5nJiBpbnB1dCkgewogICAgc3RkOjp2ZWN0b3I8c3RkOjpwYWlyPGludCwgRmFrZVNvbmc+PiBzb25nczsKICAgIHNvbmdzLnJlc2VydmUoZG93bmxvYWRlZC5zaXplKCkpOwogICAgY29uc3QgYm9vbCBlbXB0eV9pbnB1dCA9IGlucHV0LmVtcHR5KCk7CiAgICBmb3IgKGNvbnN0IGF1dG8mIFtpbmRleF9zdHIsIHNvbmddIDogZG93bmxvYWRlZCkgewogICAgICAgIGludCBzY29yZSA9IDA7CiAgICAgICAgaWYgKCFlbXB0eV9pbnB1dCAmJiAhZnRzOjpmdXp6eV9tYXRjaChpbnB1dC5jX3N0cigpLCBpbmRleF9zdHIuY19zdHIoKSwgc2NvcmUpKSBjb250aW51ZTsKICAgICAgICBzb25ncy5wdXNoX2JhY2soe3Njb3JlLCBzb25nfSk7CiAgICB9CiAgICBpZiAoIWVtcHR5X2lucHV0ICYmIHNvbmdzLnNpemUoKSA+IDEpIHsKICAgICAgICBzdGQ6OnNvcnQoc29uZ3MuYmVnaW4oKSwgc29uZ3MuZW5kKCksIFsmXShjb25zdCBhdXRvJiBhLCBjb25zdCBhdXRvJiBiKSB7CiAgICAgICAgICAgIHJldHVybiBhLmZpcnN0ID4gYi5maXJzdDsKICAgICAgICB9KTsKICAgIH0KICAgIHJldHVybiBzb25nczsKfQoKaW50IG1haW4oKSB7CiAgICBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8c3RkOjpzdHJpbmcsIEZha2VTb25nPj4gZG93bmxvYWRlZCA9IHsKICAgICAgICB7IlN0ZXJlbyBNYWRuZXNzIEZvcmV2ZXIiLCB7MSwgIlN0ZXJlbyBNYWRuZXNzIiwgIkZvcmV2ZXIifX0sCiAgICAgICAgeyJCYWNrIG9uIFRyYWNrIERKVkkiLCB7MiwgIkJhY2sgb24gVHJhY2siLCAiREpWSSJ9fSwKICAgICAgICB7IlBvbGFyZ2Vpc3QgU3RlcCIsIHszLCAiUG9sYXJnZWlzdCIsICJTdGVwIn19LAogICAgICAgIHsiRHJ5IE91dCBESlZJIiwgezQsICJEcnkgT3V0IiwgIkRKVkkifX0sCiAgICAgICAgeyJCYXNlIEFmdGVyIEJhc2UgREpWSSIsIHs1LCAiQmFzZSBBZnRlciBCYXNlIiwgIkRKVkkifX0sCiAgICB9OwoKICAgIC8vIDEuIGVtcHR5IGlucHV0IHJldHVybnMgZXZlcnl0aGluZywgdW5zb3J0ZWQKICAgIGF1dG8gYWxsID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICIiKTsKICAgIGFzc2VydChhbGwuc2l6ZSgpID09IDUpOwoKICAgIC8vIDIuIGV4YWN0LWlzaCBxdWVyeSByYW5rcyB0aGUgcmlnaHQgc29uZyBmaXJzdAogICAgYXV0byBkanZpID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICJkanZpIik7CiAgICBhc3NlcnQoZGp2aS5zaXplKCkgPT0gMyk7CiAgICBmb3IgKGF1dG8mIFtzY29yZSwgc10gOiBkanZpKSAodm9pZClzY29yZSwgYXNzZXJ0KHMuYXJ0aXN0ID09ICJESlZJIik7CgogICAgLy8gMy4gZnV6enkgcXVlcnkgbWF0Y2hlcyBhY3Jvc3MgbmFtZSthcnRpc3QgYmxvYgogICAgYXV0byBwb2xhciA9IGRvX3NlYXJjaChkb3dubG9hZGVkLCAicG9sciIpOwogICAgYXNzZXJ0KHBvbGFyLnNpemUoKSA9PSAxICYmIHBvbGFyWzBdLnNlY29uZC5pZCA9PSAzKTsKCiAgICAvLyA0LiBubyBtYXRjaCAtPiBlbXB0eQogICAgYXNzZXJ0KGRvX3NlYXJjaChkb3dubG9hZGVkLCAienp6cXFxIikuZW1wdHkoKSk7CgogICAgLy8gNS4gc2Nyb2xsIGNsYW1wOiBtaXJyb3JzIHVwZGF0ZV9zY3JvbGxfZWxlbWVudHMgY2xhbXBpbmcKICAgIGNvbnN0ZXhwciBmbG9hdCBoZWlnaHQgPSA1NS5mOyAvLyB3aWRnZXRfaGVpZ2h0ICsgc3BhY2luZwogICAgY29uc3RleHByIGludCBuX3dpZGdldHMgPSA2OwogICAgYXV0byBjbGFtcF9zY3JvbGwgPSBbJl0oc2l6ZV90IGNvdW50LCBmbG9hdCB5KSB7CiAgICAgICAgY29uc3QgYXV0byBtID0gaGVpZ2h0ICogKHN0YXRpY19jYXN0PGludD4oY291bnQpIC0gbl93aWRnZXRzICsgMyk7CiAgICAgICAgaWYgKG0gPCAwKSByZXR1cm4gMC5mOwogICAgICAgIHJldHVybiBzdGQ6OmNsYW1wKHksIDAuZiwgbSk7CiAgICB9OwogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgwLCA5OTkuZikgPT0gMC5mKTsgICAvLyBlbXB0eSBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgzLCA5OTkuZikgPT0gMC5mKTsgICAvLyBzaG9ydCBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgLTUwLmYpID09IDAuZik7ICAvLyBuZWdhdGl2ZSBwaW5zIHRvIDAKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDFlOWYpID09IGhlaWdodCAqICgyMCAtIG5fd2lkZ2V0cyArIDMpKTsgLy8gY2FwcyBhdCBtYXgKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDEwMC5mKSA9PSAxMDAuZik7IC8vIGluLXJhbmdlIHVudG91Y2hlZAoKICAgIC8vIDYuIHJlc3VsdCBsYWJlbCBwbHVyYWxpemF0aW9uCiAgICBhdXRvIGxhYmVsID0gW10oc2l6ZV90IG4pIHsKICAgICAgICBzdGQ6OnN0cmluZ3N0cmVhbSBzOyBzIDw8IG4gPDwgIiByZXN1bHQiOyBpZiAobiAhPSAxKSBzIDw8ICJzIjsgcmV0dXJuIHMuc3RyKCk7CiAgICB9OwogICAgYXNzZXJ0KGxhYmVsKDApID09ICIwIHJlc3VsdHMiICYmIGxhYmVsKDEpID09ICIxIHJlc3VsdCIgJiYgbGFiZWwoNSkgPT0gIjUgcmVzdWx0cyIpOwoKICAgIC8vIDcuIGV4cG8tb3V0IGVhc2luZzogcmVuZGVycyBlYXNlIHRvd2FyZCBhIHRhcmdldCB0aGUgc2FtZSB3YXkgdGhlIG1vZAogICAgLy8gICAgZG9lcywgbXVzdCBiZSBtb25vdG9uaWMsIGZyYW1lcmF0ZS1jb3JyZWN0LCBhbmQgbGFuZCBleGFjdGx5IG9uIHRhcmdldAogICAgYXV0byBlYXNlU3RlcCA9IFtdKGZsb2F0IGN1cnJlbnQsIGZsb2F0IHRhcmdldCwgZmxvYXQgZHQpIC0+IGZsb2F0IHsKICAgICAgICBjb25zdCBmbG9hdCBmYWN0b3IgPSAxLmYgLSBzdGQ6OmV4cCgtMTguZiAqIGR0KTsKICAgICAgICByZXR1cm4gY3VycmVudCArICh0YXJnZXQgLSBjdXJyZW50KSAqIGZhY3RvcjsKICAgIH07CiAgICB7CiAgICAgICAgZmxvYXQgcG9zID0gMC5mLCB0YXJnZXQgPSAzMDAuZjsKICAgICAgICBmbG9hdCBwcmV2ID0gcG9zOwogICAgICAgIGludCBzdGVwcyA9IDA7CiAgICAgICAgd2hpbGUgKHN0ZDo6YWJzKHRhcmdldCAtIHBvcykgPiAwLjAxZiAmJiBzdGVwcyA8IDEwMDAwMCkgewogICAgICAgICAgICBwb3MgPSBlYXNlU3RlcChwb3MsIHRhcmdldCwgMS5mIC8gNjAuZik7CiAgICAgICAgICAgIGFzc2VydChwb3MgPj0gcHJldiAtIDFlLTZmKTsgICAvLyBuZXZlciBtb3ZlcyBiYWNrd2FyZHMKICAgICAgICAgICAgYXNzZXJ0KHBvcyA8PSB0YXJnZXQgKyAxZS02Zik7IC8vIG5ldmVyIG92ZXJzaG9vdHMgKGV4cG8gb3V0KQogICAgICAgICAgICBwcmV2ID0gcG9zOwogICAgICAgICAgICArK3N0ZXBzOwogICAgICAgIH0KICAgICAgICBhc3NlcnQoc3RkOjphYnMocG9zIC0gdGFyZ2V0KSA8IDAuMDFmKTsKICAgICAgICAvLyBhIDMwZnBzIHJ1biBzaG91bGQgbmVlZCBmZXdlciBzdGVwcyB0aGFuIHRoZSBzYW1lIGRpc3RhbmNlIGF0IDYwZnBzCiAgICAgICAgZmxvYXQgcG9zMzAgPSAwLmY7CiAgICAgICAgaW50IHN0ZXBzMzAgPSAwOwogICAgICAgIHdoaWxlIChzdGQ6OmFicyh0YXJnZXQgLSBwb3MzMCkgPiAwLjAxZiAmJiBzdGVwczMwIDwgMTAwMDAwKSB7CiAgICAgICAgICAgIHBvczMwID0gZWFzZVN0ZXAocG9zMzAsIHRhcmdldCwgMS5mIC8gMzAuZik7CiAgICAgICAgICAgICsrc3RlcHMzMDsKICAgICAgICB9CiAgICAgICAgYXNzZXJ0KHN0ZXBzMzAgPCBzdGVwcyk7CiAgICAgICAgYXNzZXJ0KHN0ZDo6YWJzKHBvczMwIC0gdGFyZ2V0KSA8IDAuMDFmKTsKICAgICAgICBzdGQ6OmNvdXQgPDwgIiAgZWFzaW5nOiA2MGZwcyAiIDw8IHN0ZXBzIDw8ICIgc3RlcHMsIDMwZnBzICIgPDwgc3RlcHMzMCA8PCAiIHN0ZXBzXG4iOwogICAgfQoKICAgIHN0ZDo6Y291dCA8PCAiQUxMIExPR0lDIFRFU1RTIFBBU1NFRCIgPDwgc3RkOjplbmRsOwogICAgcmV0dXJuIDA7Cn0K",
 "about.md": "PiBUaGlzIG1vZCB3YXMgcG9ydGVkIHRvIGdlb2RlIHVzaW5nIEFJLiBJZiB5b3UgZG9uJ3QgbGlrZSB1c2luZyBBSSBwb3J0ZWQgbW9kcywgZmVlbCBmcmVlIHRvIG5vdCB1c2UgaXQuCgojIFNvbmcgU2VhcmNoCgpBbiBpbi1nYW1lIHNlYXJjaCBkaWFsb2cgZm9yIHlvdXIgZG93bmxvYWRlZCBzb25ncywgd2l0aCBmdXp6eSBtYXRjaGluZywgc29ydGFibGUKZmlsdGVycywgYW5kIEp1a2Vib3ggTk9ORyBzdXBwb3J0LgoKVGhlIHNlYXJjaCBkaWFsb2cgcmVwbGFjZXMgdGhlIGRlZmF1bHQgc29uZyBicm93c2VyIGluIHRoZSBlZGl0b3IncyBzb25nIG1lbnUuIEFzIHlvdQp0eXBlLCBzb25ncyBhcmUgcmFua2VkIGJ5IGhvdyB3ZWxsIHRoZXkgbWF0Y2gg4oCUIG1hdGNoaW5nIGFjcm9zcyBib3RoIHRoZSBzb25nIG5hbWUgYW5kCnRoZSBhcnRpc3Qg4oCUIHNvIHNob3J0LCBhcHByb3hpbWF0ZSBxdWVyaWVzIHN0aWxsIGZpbmQgdGhlIHJpZ2h0IHRyYWNrLgoKIyMgRmVhdHVyZXMKCi0gKipGdXp6eSBzZWFyY2gqKiBhY3Jvc3Mgc29uZyBuYW1lIGFuZCBhcnRpc3QKLSAqKkZpbHRlciBidXR0b25zKiogaW4gdGhlIGNvcm5lciBvZiB0aGUgZGlhbG9nLCB0byBzd2l0Y2ggd2hhdCB5b3UncmUgc2VhcmNoaW5nIGFuZAogIGhvdyByZXN1bHRzIGFyZSBvcmRlcmVkOgogIC0gKipQcmlvcml0eSoqIChkZWZhdWx0IG9yZGVyaW5nIGZyb20gR0QpCiAgLSAqKk5hbWUqKgogIC0gKipBcnRpc3QqKgogIC0gKipTb25nIElEKioKICAtIHBsdXMgYW4gKiphc2NlbmRpbmcvZGVzY2VuZGluZyoqIHRvZ2dsZQotICoqU29uZyBwcmV2aWV3Kiogd2l0aG91dCBsZWF2aW5nIHRoZSBkaWFsb2csIHdpdGggdGhlIHBsYXkgYnV0dG9uIHVwZGF0aW5nIHBlciByb3cKLSAqKkRlbGV0ZSBzb25ncyoqIHN0cmFpZ2h0IGZyb20gYSByZXN1bHQgcm93ICh3aXRoIGEgY29uZmlybWF0aW9uKQotICoqSnVrZWJveCBpbnRlZ3JhdGlvbioqOiBzb25ncyB0aGF0IGhhdmUgYSBOT05HIGNvbmZpZ3VyZWQgc2hvdyBhIGRpc2Mgc3ltYm9sLCBhbmQKICB0aGUgTk9ORydzIG5hbWUvYXJ0aXN0IGNhbiBiZSBzaG93biBpbnN0ZWFkIG9mIHRoZSBvcmlnaW5hbC4gVGhpcyBpcyBlbnRpcmVseQogIG9wdGlvbmFsIOKAlCB3aXRob3V0IEp1a2Vib3ggaW5zdGFsbGVkIG5vdGhpbmcgY2hhbmdlcyBhbmQgdGhlIG1vZCB3b3JrcyB0aGUgc2FtZS4KLSAqKlNldHRpbmdzKiogZm9yIGV2ZXJ5IHBhcnQgb2YgdGhlIGFib3ZlLCBpbmNsdWRpbmcgZnVsbHkgZGlzYWJsaW5nIHRoZSBtb2QKCiMjIENyZWRpdHMKCi0gKiptYXRjb29sKiog4oCUIHRoZSBvcmlnaW5hbCBtb2QsIGluIFtzbWFsbC1nZC1tb2RzXShodHRwczovL2dpdGh1Yi5jb20vbWF0Y29vbC9zbWFsbC1nZC1tb2RzKQotICoqQWxwaGFsYW5lb3VzKiog4oCUIFtJbXByb3ZlZCBTb25nIEJyb3dzZXJdKGh0dHBzOi8vZ2l0aHViLmNvbS9BbHBoYWxhbmVvdXMvSW1wcm92ZWQtU29uZy1Ccm93c2VyKSwKICB3aGljaCB0aGUgZmlsdGVyIG1vZGVzIGFuZCBzb3J0IGJ1dHRvbnMgYXJlIGJhc2VkIG9uLCBhbmQgd2hvc2UgaWNvbnMgYXJlIHVzZWQKLSAqKkZsZXltIC8gRmxhZnkqKiDigJQgW0p1a2Vib3hdKGh0dHBzOi8vZ2l0aHViLmNvbS9GbGVleW0vanVrZWJveCkgYW5kIEF1dG9Ob25nLCBmb3IgdGhlCiAgTk9ORyBzdXBwb3J0CgojIyBQb3J0aW5nIG5vdGVzCgpUaGlzIGlzIGEgcG9ydCBvZiB0aGUgb3JpZ2luYWwgV2luZG93cy1vbmx5IE1pbkhvb2stYmFzZWQgRExMIHRvIGEgcHJvcGVyIEdlb2RlIG1vZC4KVGhlIG9yaWdpbmFsIGhvb2tlZCBgQ3VzdG9tU29uZ0xheWVyOjpvblNvbmdCcm93c2VyYCBhdCBhIGhhcmRjb2RlZCBvZmZzZXQ7IHRoZSBwb3J0CnVzZXMgR2VvZGUncyBgJG1vZGlmeWAgaG9vayBpbnN0ZWFkLCB3aGljaCBpcyB3aGF0IG1ha2VzIGl0IHdvcmsgYWNyb3NzIEdEIHZlcnNpb25zCmFuZCBwbGF0Zm9ybXMuCgpUaGUgTk9ORyBpbnRlZ3JhdGlvbiBkZXNlcnZlcyBhIG5vdGU6IEp1a2Vib3ggZG9lcyBub3QgZXhwb3NlIGEgcHVibGljIEFQSSBvciBidW5kbGVkCmhlYWRlcnMsIHNvIHJhdGhlciB0aGFuIHJlaW1wbGVtZW50aW5nIE5PTkcgc3RvcmFnZSwgdGhlIHBvcnQgcmVhZHMgdGhlIGRpc2MgaWNvbiBhbmQKbWV0YWRhdGEgdGhhdCBKdWtlYm94IGl0c2VsZiBoYXMgYWxyZWFkeSByZXNvbHZlZCBhbmQgcmVwYWludHMgdGhlbSBpbiB0aGUgc2VhcmNoIHJvd3MuCgpDb250cmlidXRpb25zIGFuZCBidWcgcmVwb3J0cyBhcmUgd2VsY29tZSDigJQgaWYgc29tZXRoaW5nIGxvb2tzIHdyb25nLCBpdCdzIGxpa2VseSBpbgp0aGlzIHBvcnQgcmF0aGVyIHRoYW4gdGhlIG9yaWdpbmFsLgo=",
 "resources/artist_icon.png": "iVBORw0KGgoAAAANSUhEUgAAAEgAAABcCAYAAADNqvPKAAAABGdBTUEAALGPC/xhBQAAACBjSFJNAAB6JgAAgIQAAPoAAACA6AAAdTAAAOpgAAA6mAAAF3CculE8AAAAGmVYSWZNTQAqAAAACAABARIAAwAAAAEAAQAAAAAAABPAdecAAAAJcEhZcwAACxMAAAsTAQCanBgAAAM3aVRYdFhNTDpjb20uYWRvYmUueG1wAAAAAAA8eDp4bXBtZXRhIHhtbG5zOng9ImFkb2JlOm5zOm1ldGEvIiB4OnhtcHRrPSJYTVAgQ29yZSA2LjAuMCI+CiAgIDxyZGY6UkRGIHhtbG5zOnJkZj0iaHR0cDovL3d3dy53My5vcmcvMTk5OS8wMi8yMi1yZGYtc3ludGF4LW5zIyI+CiAgICAgIDxyZGY6RGVzY3JpcHRpb24gcmRmOmFib3V0PSIiCiAgICAgICAgICAgIHhtbG5zOnhtcD0iaHR0cDovL25zLmFkb2JlLmNvbS94YXAvMS4wLyIKICAgICAgICAgICAgeG1sbnM6dGlmZj0iaHR0cDovL25zLmFkb2JlLmNvbS90aWZmLzEuMC8iCiAgICAgICAgICAgIHhtbG5zOmV4aWY9Imh0dHA6Ly9ucy5hZG9iZS5jb20vZXhpZi8xLjAvIj4KICAgICAgICAgPHhtcDpDcmVhdG9yVG9vbD5BcnRzdHVkaW8gUHJvPC94bXA6Q3JlYXRvclRvb2w+CiAgICAgICAgIDx0aWZmOlJlc29sdXRpb25Vbml0PjI8L3RpZmY6UmVzb2x1dGlvblVuaXQ+CiAgICAgICAgIDx0aWZmOlhSZXNvbHV0aW9uPjcyPC90aWZmOlhSZXNvbHV0aW9uPgogICAgICAgICA8dGlmZjpZUmVzb2x1dGlvbj43MjwvdGlmZjpZUmVzb2x1dGlvbj4KICAgICAgICAgPHRpZmY6T3JpZW50YXRpb24+MTwvdGlmZjpPcmllbnRhdGlvbj4KICAgICAgICAgPGV4aWY6UGl4ZWxYRGltZW5zaW9uPjcyPC9leGlmOlBpeGVsWERpbWVuc2lvbj4KICAgICAgICAgPGV4aWY6Q29sb3JTcGFjZT4xPC9leGlmOkNvbG9yU3BhY2U+CiAgICAgICAgIDxleGlmOlBpeGVsWURpbWVuc2lvbj45MjwvZXhpZjpQaXhlbFlEaW1lbnNpb24+CiAgICAgIDwvcmRmOkRlc2NyaXB0aW9uPgogICA8L3JkZjpSREY+CjwveDp4bXBtZXRhPgqIoGIJAAANnUlEQVR4Ae2ceaxVxR3HBxCQTWS10AKFx5YCoWIpIZRUGyzGttiQgA22MUbAtlHa/lH0D6NoalqoxSCmCzTYhk1sEZBNcSkREA0i0rIvsm+y7yjI6fczvnMyZ+499917ffe9ex/8km/emTkzc37ne34z85vfzH3GXJfrDFxnoIAM1Clg2+mabqvMWkJDIRA+F4paULYqpL8e8oDQWGgi3CwsEyYIV4SiJ0o6FkR6q9WZAtYCEfx1MUXp7wvXnHTVG08VXDLSXZ9XmY1Ca+GakK/pLZ8X0pGRKe9PNZ2dFnrBPwhXKyKnWbNmSUR1rIkkNdBLPSnQVZJe3Oa3b98+mDx5cnD48OGgXr166cr+paYRNFYvdLQiYlq2bBlMmDAhcOXxxx9PRxB536gJJD2sl9hbETGNGjUKnnrqqeDChQsuN/b69OnTQdOmTdOR9I9SJogBmMH0pJDu5WxerVq1gkcffTQ4fvx4CjFuxjPPPJPUxrdKkaTaUvpnwrZM5IwZMybYt2+fy0Pi9eXLl4M2bdqkI2lOKRLELLUjiZyRI0cG27ZtSyQj6cZzzz2XjqBLes63S42kH6UjZ8SIEcFHH32U9P5p869cuRLL79Spk0/SET3rF6VEEOu1PsJxIXqZYcOGBZcuXQp44atXr1rE3txLnDt3Ljh69Ghw7NixgEE6rDt+/PioTaf9n+q6rlDtku1i9W5pOlHo5mr83nvvmbKyMlO/fn1Tt25dc+ONN7q37fX58+fNqVOnjGYzIyJN7dq1jfwg89lnn5mpU6eaiRMnms8/T1mrTlflB4XLKQ1WcUa24Y5j0uu3QiNXv7Nnz5q+ffval+eFmzRpYjSL2SIQ8sknnxhZjJH1GFmMJYX8KVOmmIceesgsX77cyNDcJsNrFrkHBGZMUG2SLUEMnE2E77qabty40fTv3980bNjQfPrpp/YWFiJv2ZIDGRCjGcuoK5qZM2easWPHWmLIq0BwGll+zK+gXEFvZ9vFUALrwUFsTiKUO++80zz22GOmTp06tpvdcMMNthuFlkG3mjt3rpkxY4Y5dOhQWC3bv1dU8H5hVrYVKrtcthbEc/nkEDqIRCgff/yxue2224wWoXYsgRDI4e/ixYvNE088YZYsWWK7WVgnh7/4X2XC33KoU61FIWi/EJt5BgwYECxdujRYtmxZ8NZbbwXjxo0LunXrFivj1/HT9957bzBp0qSkOg+qfMnIGGma8iLPPvts8PTTTwc9e/ZMuZeufJg3ZMiQYMWKFZFTcNddd6Wrv7Vk2ClXNGXJcdNNN6V7scS8QYMGWYuLmCm/ePvtt5PqPFJKJI0MLSDXv3THBQsW+LzE0kOHDk1HEhNESclSaRvzrpVO92I279Zbbw1mz54dIyIpsXr16qR2iEGVjPSVpr8Xkl7G5nfv3j2YNm1aEheJ+ffdd1+6dnFYG5YMQ1IU53GukPIyhFhfeOGFRAIqurF+/fqUNsuf86T+lozgp/xAiL3M8OHD7SJWa6yKeMh4f9SoUbF2y59D/JvwS8nIUGkaixX16NHDRhTPnDkTaAliycrIRMJNYkxqOx3GVxU7uXjSSTo10437hWghq7CGUczZKNZj12h41Q0aNEiqn5jfokULc/LkSfP+++/7ZQYoY5pwxr9RrOlFUiz2pdu1axesWbMmYCzBErTyT7CTzNn79+8PtACOtV3+rOergozKsCD0ZPnxgKuwupe1mq5du9p1mWgwN9/MmYXcRA6ouXjxolm1apVfsZ8yZgu4GiUh/5KWsS+tLhK8+eabgV7OhmbVXTKbS8JddkjYRvLbV/rvhWamsiwIPXcLo7gIhS9PpFHrMxtIw4qaN49FS8KiGf8yfhFPIsDmSR+lFwiHvfyiTf5TmsW+dOPGjYNFixbZBem6deuCEydOJNhJ5myRHbRq1SrWdvmzZhaSjcq0IPTcLvzSVZhQLFFGLTWsFTGjMTvlKgTikDfeeMOv2ksZHMZiHCwJ+au0jH1pdbNAUUVrRR9++KHd2chsL8l38dD99pWeVyhmKtuC0HOT8CtX4TDKSOQRayKtQw1ukayv2REhQulJd6VXCLu8/KJNskWU8qVnzZoVWZF2PJLNpII7LIDTtP9aIdgohAWh53rh1wJrtUjYyejXr19sLAq3iaJCWVywi/Lqq6/6JTsrY61AMK/SpFAEnZOG7CIOdDXdunWruf322+0yBGIYeOXfuEWyumbAhyC2lzxpp/SLXt6XShaKIJTiazKj1ScRCvtkiipaK8K3wS9iXMpVWOuxneRJB6UZA0GlSCEJYieR9r/narpz50672chUDzHsp8lXcotkdY3zKS/d6LiNX76jMqb6mfmmC0kQOrEMHy3EGGAreuDAgdaK2JdnTw2ichVIfumll/xqbZWxW2Ac/NKSu1a5PZJTCZx8HexW2717t+nTp4+55ZZbrBVhSezr5yqazewils1LT7oq/WcvL69koQlCqdXC/QJxo0h0BMbccccdkRWx0g+95ahQFhc6pWamT5/ul2yljCPCB/6NXNNVQRA6MR790FVOcR6jyKPhBeleWBGhjVyF4zfyzo1iTn7VHsqY5Gfmmq4qgviSwwW+bCQKYxgOPyB41xDE6j9XUXDOvPhiyuyOxZ4VsOC8paoIQkGC7T92NeW0R+fOnQ0viAUBpu9cReszs2XLFsNxHE++qTSRx5QTWl65xGRVEsSsQjdjlomEQ1aDBw+OvGsG63ysiPg3B7M8aaw03XuFl591sioJQqlTwjBXOwL8WEDHjh2tBXEvn9AsY9mePXuMDpW6zXPNuWt+5gBROUvuLmzOj4hVICyb8jXxZfCqOaXGeUb8pHyE02tphJE/7Y00ZVOyqtqCUOCoMMLVRFFG6xN16dLFWhHrtHysSBFHg0VqN8Vtnuv+At51zsxXtQWh7CLhdS5cCa2ICCQ+EgdE85EEKyIcmZcVVYcF8d4HBZzHSNgmYgbDOw5DICxBchUsjy6qEyJ+VaxoulCtp2Z9pTKlCZMGLlq3bm2P8K1cuTIgwK/xqILQWfrbR44cCXR2O9Z2+XMIB+ck1WVBKLlXeNDVlkPnHEbHw8YnQvLZJiLGRFd955133Oa5Zkb7t8A4WBLClk3sS6ub2QOhnFv8MttEbHWru8XaLn8WW1NZS3VaEEqyDP+5qy1TPYvW3r1727FInSivbSKC+9TVqVu3ea57C0sExsGSELaPY19a3SyYP39+FOCv6Md56UeiwB67adu2bazt8mfhj2Ul1W1BKLlFeMTVFqcRIWbEWJTvNhF1scbXXkvZ8OBnDv8R9vCcUhAWlLEvrak+mDNnTmRF/JQqX1FIJNZ2+bPwxyqUYrAglNwg/MbXlnAsvybCL8KKCLGGPpJfNlOamVHnA/wiOFmrhH3+DTddHZ60+/zwGiUnhInw77x582xQnv00fjnEMiIfGT16tOnVq5df9YwyvuJn+ulisSD0Wic8LMQiZsxq/OQKy8Gi8t0mwsfyDj4017PwiThwQdw8rRSLBaHccSHFitiH37Fjh/3NGXtq/EAvV2G6h2AvpEsAr71wOVN7xUQQev5RgKiYhAtZrIlulsWP8WL1IYi4N961I410fUDI2IuKjaALUjjFinD2CKdCDFaU61gUug1p9t7Yqk7sXroXP1xARhEIBKXMLJryLUEc62NPHmvKViCGesyEjlzS9S7hi5NZzg33stgsKNSNrhYTrfDN2rVrrQURzti7d6/fZWLl3QRdi6meQd4RDld0Er7wSp0b7mWxEjRZSm5zFeUaK2K6J6C2a9cuO3gTovVe3K9mNwEo5/lQTPM7hYxjUMabKU+q2oyLetwQ95FsE7FdjTXwskzdgC7EDBXOVm4drhm3NmzY4J9Mq69ba4R3KZMkxUwQfhH7aDFnjq5FOLZDhw52HGJsocsxELPugjxfIHDhwoWGf4TgjEOnVY6g3Xq/vJsuZoLQk8D0PUI0FEAG29YHDx6MHEfGGEjDmjhTDVGQEloUAzqzoLdoraV2Fwt8iEQpdoL+J83LBLZuWDtZwVqYyTZt2mTHF6wGQiACkrhPVBFrIZ+NyFdeecXumYVTvhqC/JUC3SxRip0gFP+vgPt8t8CqnC9vrQO/CGs6cOCAPYTFOMR4Q7fDeiADgii3efNm34LqqRkOF7FQTpRSIOiktN8sELtB3y5CJFiNgvS2y0EI1sQ/W6HbMZDT3dgtefnllyNns7zyCf3FelJ+axU1rotSIAh9WSNsEthP+6rQQmgiRMK2EVM/ZDHT0a1CQCJH/15/PbYd11CVpwu4EzEHKWpUF6VCEDrj0J0TPhAOCWVCSyESrIaxCXeAbgZRjEWAU7GcAHHGoL2quFwo6UE6ennngul5bTna6i8WFQuRsOJnzIEwzlTT5bZv327efTfm8jRVvT8JkB1bgygdSSlZUKR0+QUrcbZP6XrfERh0I3eAGYwBnJNnjEv4T5Dk+EH7VJ7xh5myRgvW8xNhhsDUnRJ/ZsdW4Vo/n8G/p2BnRf2t0UJPYDx6XtgqEATzCfHTK1Smm3BNSRu97T3CPMEnxE0f1f3RQsZQh+7XWPm63ux3AjOeb02MPWOErLpWVoXUWKkJgzV+TldhnMDMVyYsFOiG+FUQd80LRNGN6HrhLFdTjeKa/9jXCShJBv4PiJifWJHOGFYAAAAASUVORK5CYII=",
 "resources/id_icon.png": "iVBORw0KGgoAAAANSUhEUgAAAFwAAABcCAYAAADj79JYAAAABGdBTUEAALGPC/xhBQAAACBjSFJNAAB6JgAAgIQAAPoAAACA6AAAdTAAAOpgAAA6mAAAF3CculE8AAAAGmVYSWZNTQAqAAAACAABARIAAwAAAAEAAQAAAAAAABPAdecAAAAJcEhZcwAACxMAAAsTAQCanBgAAAM3aVRYdFhNTDpjb20uYWRvYmUueG1wAAAAAAA8eDp4bXBtZXRhIHhtbG5zOng9ImFkb2JlOm5zOm1ldGEvIiB4OnhtcHRrPSJYTVAgQ29yZSA2LjAuMCI+CiAgIDxyZGY6UkRGIHhtbG5zOnJkZj0iaHR0cDovL3d3dy53My5vcmcvMTk5OS8wMi8yMi1yZGYtc3ludGF4LW5zIyI+CiAgICAgIDxyZGY6RGVzY3JpcHRpb24gcmRmOmFib3V0PSIiCiAgICAgICAgICAgIHhtbG5zOnhtcD0iaHR0cDovL25zLmFkb2JlLmNvbS94YXAvMS4wLyIKICAgICAgICAgICAgeG1sbnM6dGlmZj0iaHR0cDovL25zLmFkb2JlLmNvbS90aWZmLzEuMC8iCiAgICAgICAgICAgIHhtbG5zOmV4aWY9Imh0dHA6Ly9ucy5hZG9iZS5jb20vZXhpZi8xLjAvIj4KICAgICAgICAgPHhtcDpDcmVhdG9yVG9vbD5BcnRzdHVkaW8gUHJvPC94bXA6Q3JlYXRvclRvb2w+CiAgICAgICAgIDx0aWZmOlJlc29sdXRpb25Vbml0PjI8L3RpZmY6UmVzb2x1dGlvblVuaXQ+CiAgICAgICAgIDx0aWZmOlhSZXNvbHV0aW9uPjcyPC90aWZmOlhSZXNvbHV0aW9uPgogICAgICAgICA8dGlmZjpZUmVzb2x1dGlvbj43MjwvdGlmZjpZUmVzb2x1dGlvbj4KICAgICAgICAgPHRpZmY6T3JpZW50YXRpb24+MTwvdGlmZjpPcmllbnRhdGlvbj4KICAgICAgICAgPGV4aWY6UGl4ZWxYRGltZW5zaW9uPjkyPC9leGlmOlBpeGVsWERpbWVuc2lvbj4KICAgICAgICAgPGV4aWY6Q29sb3JTcGFjZT4xPC9leGlmOkNvbG9yU3BhY2U+CiAgICAgICAgIDxleGlmOlBpeGVsWURpbWVuc2lvbj45MjwvZXhpZjpQaXhlbFlEaW1lbnNpb24+CiAgICAgIDwvcmRmOkRlc2NyaXB0aW9uPgogICA8L3JkZjpSREY+CjwveDp4bXBtZXRhPgqnJhWXAAANxElEQVR4Ae2caYgVyxXHy3XcjUaNu2NwQRGJohI1mifG8HyRgA8fopFk1EgIiUSJBiFGRf0iiEoQNIqKEX2SiMa4BN+X98EFDeKC+77v6+i4L53z69yaqe6uvrd75vYdP/SBc6ururrOqX+dOlVdVX2VSilFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEWgGhCoUQWZdeTZhsJNhRtkrhtJWFv4g/Bb4XfCZcKPhZ8KvxbOB9WUQoqEkds4w/UzcXRCN/SolQlfSfhcGF1eZkL0KRV+L/xJUhPR6ifCK4QvCjuV5IPy3BfCgBGFvieZkDtf+BvhB8KVlW177r6U9y/hmcJ9hGnIaqWRIv2AsE3ZqqZ9GVKz1pL+a2GssqoyKvP8cZH7K+F2wgUjXMVK4cooHOeZ9SIDFwAR/lk4zvNJ5/2v6PMzYdxYYvSZlJx0Rczyj4o8wD5SYLmmDlGufyH6VWXcsz78mRT6rXBWaty4serdu7fq2LGj4lpz7dq11YcPH9Tbt29dfv78ubp06ZK6cOGCun79uvr48WNYuf+UG1+F3TTTW7Zsqdq3b68IW7RooZo1a6YaNGigGjZs6IZ16tRRtWrVKuc3b96osrIyl1+8eKGePn2qLl68qM6fP68ePnxoFh3l+rRkmiH8nyiZc+XpKhmsLV2jRg1nyJAhzpo1axwB0BHgnLhUWlrqzJ492xFwrDJsstu0aeNMmTLFlXv06FFHGjCu2Kz5Hz165Bw4cMBZtGiRM3z4cKeoqCiqbstF3yoNsHSVE7ZKDx061Dl79mxWxePcvHXrltO1a9esFevSpYuzfft2R3pLnKKrnFd6gLNr1y5n9OjRjvSSrDoKVueEmUVVisbKUwEB8+bNS6TSe/fudWrWrBmQhw5NmzZ1aJTqpmvXrjmzZs1yxGVZ9TTw6hsXcaz7hVGAK2DixImJ1nnChAnWiowbNy5RuXELf/DggTNp0iQHt+rHyIj/SK4jUzfJ6SmsSZMmDj43SVq2bJlHptZh8eLFSYqtdNn79+93evbsadU5o3tO0PXc8uf+ppGBSgno/uS8xrt1o52DJP49mPgJpAwaNEgdOnRIjRo1KkybvXKjOOymmR4YLGnNpOnEiRNWazly5EjSoqtUPgP59OnTrboLqKwj8dIYSli55+F69eo5MnetklJRHg4D/Pbt21Eer/Y8THH92GXiGyUMfUFilc3zYJ8+fQpSmePHj3vkaj1ev35dqXl+QZT2CRk/fry1DlIX60sc1s1ypodatWrliScVsb118rZYt25dV6TULSnReSt33bp1Sl4IbeX9QxID2AI468ke4nUZMJKusA1wlgiQm7RsXeGXL18qedtUjx8/VlzHlYtxbNy4UTVqxFZAgOb4U1ikZ9HIQ6xFaDBYk5D5p+d+viKsufgpRHF/tirH3717p+7cueOupbDOQpx61q9f312nad26tQKHKNShQwe1cOFCNW3aNH/2P0kCSwA39A0snJ0ZD7HYoy08bot7CsoR0Y1qZgPwpC0cuffv31fPnj1TLGaxwMaCFlYuLznq3LlzSmZK7nXU+k+dOlX169fPrIq+9ky5ATywxSSDVnmlowrUpccJbYDLDKm8iKRkUz8tGzeybds2tWfPHnXmzJnyBqAxLl++rK5cueJaf7lSIReyTOFaueU2Fl6+u4VLYd/RQ1h4UpU1BelKm2kAjuyk3BiykIu7gGfOnKlkelquQo8ePdSYMWPUgAEDFA0D8PSGdu1yb/6MGDHCXa5mGdpHP5W4u5xrtXAALwTZfLgsjyYqmsbEGhnsGCxNsBGMlS9YsEDJcq27ro9vx8fLMkdOvSi3pKTElu8vOjEr4FhZkpZus3ANeJJy2SRBDhsQYbRv3z61YsUKt6ehJ4ADfi6SBT9b7xwoz32XZwE8MFVgt6YQZAMcMJIEGyNi5oWFM1Bmoy1btihZk3cBRCf8fS4qLi5WspFhy+YubAF44O2Crp6kD9Xa2AAHjKQJGTQsLiUXbdiwwW0g8Ig6tk2ePNlW7BgSrYCbQCQJvM2HA0QhiHpF2c+UTQglSxDloEdxK7JDZqtCLxKtgCfZpU1NzIbV6YUCHHkhAyEHgjy0detWF3AGRZuReDJLhJcmvTxh3PsB1wAeIBsQgUx5SLDJoVJQIRo9ZKz6t79qvARhCLiiKHpRB94+LVRktfAorWgpLHaSDXC6epRKxRZmeSAE8MD8j3m4Bjyqi+3UqZNFompsBdwEIsnK2xo2aoVstYmbFuKPmbWVmWU9efLEdSVYeFT9QgBvZXUpSYJsVsRsWJ2uXYqOJxmGWDiAV7x6ZhRgRgPgUfULA5wpQWBaCOC0ZNTWrCwoNsCTlmnqauthGTw8Fs4zvObHsXCWuC3UqjBzMItkkmyAF6p3IT9kRkSvd2cU5NHUuXNnF3AdzxWyCmmhsmoF3GZhhQTcMnUDI968WppgYa1sjMQhFr0sVGr14ZaMiSRVt4WHbDAEXnWx7rgUAnj1WrgNcNL0GBK3knHzhwD+Q385UQFHb12nmzdv+osh/qhaXYpWztRMuxQdmvfyfR3iUhb45fTv39+fZI1TH20w/mXfzAOl1Qq4zYe/f1+xAZU06HJo1Aacu+ahb7AhUlJSoqNZQ+oD4K9evXJ3iyyZPz2XYgJuUTivSc2bN89Z3tixY91D/7kyArQGnMUu4j46JHEndNDEupK2MItSCsCTlquByAU41j1jxgydPWuI3gAO796925Z3G4nV6lLCALdpm0RaLsBXr16tevXyeJhQNXhrBXTqtGPHDlu+7SSGAo6VJf3WhzX4Se+nFqKH8Y2SjXijnDNnjpLz67bbgTTAZl2G+vAtk3wtEsgjCRdItLoUWklXOMnubbNwDbhN43yn9e3bV8nh//JimbWMHDnS3XAA8KjE1huAwytX8qVlgP4qKa51YeEBM2OXGkoSbMq3Aa7PxCTdu5CPjE2bNqm5c+e6X8C1bds28uIUz0MYCDpj3Xfv3lWbN2/+/w3v79c6CuCB4ZSTSBpsHeoH8hmGAY4M5CYp26xH9+7dzWjka3w2p7W0O1m+fLn7qaSvgBsS53N3lwA8sFpIFwEMfFmSlbb5cHpXkjIz9c5LwDo5Fg7wTAXZcLbQVDNN+/B7ZiJgA3rSVmazcO3O0OdTBp4NaDwBAyY6c4LLYkCPpRqeLTsN+CUTcK4psDoA1+4sadn++saJgw2uBOvGncyfP989g2gp45eS5vEgGvDT/szy2YfrVrDCpCyNwcZPrCNTEWTaeoA/fyHjgMuiFHuc6I6eS5YscQ+DWvRgurLLn67n4YEzX/JhqttFdMXx5/kg03JR3EZYkHzy7d4ifyFmLDY9zDTWR8AEgwB49Fq/fr1au3atmU1fY8B/0BEz1IBfNRO5piUpmMoyP80X4PptDBlMo2zEQXn+uADZ1Q04PhqgOcPCNX6acOnSpe4xOJv+kjZa2HpeUAPO1MVDjLq6y7Nxap7b9mSMGaFMXAVgyp8VWJ/mTDbfcAI2cvMl2yosJBF3weH8e/fuuQBrQ2EzmXn7yZMnQ55UHNwMeAydWQN+USfo8NixY+5MBXCYsbBYH3ebSZelQ8phUARsrPjq1av6lic8ffq0Gjx4sGtN7Dtq9mRKIIL16vPgTPno4QBNOix/eqD4iMqcSfnU+Fzi5XNu3z03qgEPnGrk/0SwQHY7EIZf5XMQvnDj/0niWB1WTUUY2QEb98TCUBjxtS/LonRdrIxGZ+0a5utoZHPcOGQTOKzYQDqAohtGAMAwAGtGLr2MM+OrVq1y10kChVQkAPaeiqj9yvxa6oBkoTuUk/yFhpKPP12QUUL7UwDj4yMagIrD+HnAhGggKkO31IMMbonnyMcMCEC1yyoXmLkgH1Mt1jooQ8vmNuWYjB7IpwdqNu+jE8DB2lLRDZDNdH2fkHoi8+DBg2rnzp1Zz5GLSvhqdvnPCOckE3D+TCywkEtlAIcToVyjkCYqA+sKApROM/Nwn3s8f+PGDSV/iZHz5CrlDBw4UA0bNsztZYCQTbbWQeuDfK0LoZ8oD4Z0SIOw2sf4xYH8CMeZt8jjk4WtW/SU7Se/Jsslw+/8mYhjmeztyb8puN+xmNtTupKAalacdNKwJr4MO3z4sGs1uoI2ObY0PpYtLi52D0gye2EdG+Zvl/zyiJtA+68pnzR04EA+gyLjCV9DnDp1yt0es+lgSRsvaZuFPS82lnyeJD/gfAZ+TriDJ5clgi+l0oChWVs4roKBhakUFQo5FGOW+neJvBL+jZmY6xpw9X9dYRD0IBjfji6acQ+MB7gSmDk1YwNplaC/yTN/FLae9KlEeQrQ1wvTcoXgb0SOfuOlhxVCZmVkLBXd7P83IjfyQV9KIZVRLM4zAKzB1jozljCAxyknyby/FV3cD6K0gkmGRVI4wDMC57NSAAqw2ej7cvP3wneF8yk7V1lXRd404S7CfmOQpKqR34dnK43zdj8W/kL4K+FGwlHpjmT8Vvhr4f3CT4TjUAPJXCxMl2ZXt08mrGoXZ/ORxj8kzBHlU8KRZxySNzbFAdxfOF/+1xfmLyqaCn9HWFsESj8VJnwpXHG6RyJ5JmSiC1xXmF7JNem82LHqxgeWLE0ySnL9JnNdMceVhJRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBFIEUgRSBAqHwP8AzHd9s3gx0lAAAAAASUVORK5CYII="
}
ROOT = pathlib.Path('/kaggle/working/song-search')
for rel, b64data in FILES.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64data))
    print('wrote', p, p.stat().st_size, 'bytes')

# the SDK requires mod.json "geode" to match its own version exactly, and
# compares major.minor with the "v" prefix stripped from both sides, so the
# field must carry no prefix
sdk_ver = pathlib.Path('/tmp/work/sdk/VERSION').read_text().strip().lstrip('v')
mj = ROOT / 'mod.json'
m = json.loads(mj.read_text())
if m.get('geode') != sdk_ver:
    print('mod.json geode', m.get('geode'), '->', sdk_ver)
    m['geode'] = sdk_ver
    mj.write_text(json.dumps(m, indent=2) + chr(10))
print('mod.json geode =', m['geode'])

In [ ]:
%%bash

set -eo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "4. mod source + native logic test"
cd /kaggle/working/song-search || exit 10
find . -type f | sort | tee -a "$LOG"
g++ -std=c++20 -O2 -Wall -I src -o /tmp/work/logic_test tests/logic_test.cpp
/tmp/work/logic_test | tee -a "$LOG"
log "logic test rc=$?"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "5. configure (cross-compile -> Win64)"
cd /kaggle/working/song-search || exit 11
export PATH=/usr/local/bin:/tmp/work/geode-cli:/tmp/work/xwin:$PATH
export SPLAT_DIR=/tmp/work/splat
export TOOLCHAIN=/tmp/work/toolchain/clang-cl-msvc.cmake
export HOST_ARCH=x86_64
export GEODE_SDK=/tmp/work/sdk
export GEODE_BINDINGS_REPO_PATH=/tmp/work/bindings

rm -rf build
cmake \
  -DCMAKE_TOOLCHAIN_FILE=$TOOLCHAIN \
  -DCMAKE_BUILD_TYPE=Release \
  -DGEODE_TARGET_PLATFORM=Win64 \
  -G Ninja \
  -B build > /tmp/work/configure.log 2>&1
RC=$?
tail -50 /tmp/work/configure.log | tee -a "$LOG"
echo "configure rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 12

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "6. build"
cd /kaggle/working/song-search || exit 13
export PATH=/usr/local/bin:/tmp/work/geode-cli:$PATH
export GEODE_SDK=/tmp/work/sdk
time cmake --build build --config Release > /tmp/work/build.log 2>&1
RC=$?
tail -70 /tmp/work/build.log | tee -a "$LOG"
echo "build rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 14
log "BUILD DONE"
find build -maxdepth 3 -type f \( -name '*.geode' -o -name '*.dll' -o -name '*.lib' \) \
  -exec ls -la {} \; | tee -a "$LOG"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
cd /kaggle/working/song-search || exit 15
GEO=$(find build -maxdepth 3 -name '*.geode' 2>/dev/null | head -1)
if [ -z "$GEO" ]; then
  echo "NO .geode PRODUCED" | tee -a "$LOG"
  find build -maxdepth 3 -type f 2>/dev/null | head -30 | tee -a "$LOG"
  exit 16
fi
echo "geode: $GEO ($(stat -c%s "$GEO") bytes)" | tee -a "$LOG"
unzip -l "$GEO" | tee -a "$LOG"

log "7. DLL architecture check"
mkdir -p /tmp/work/geocheck
cd /tmp/work/geocheck
rm -rf ./* 2>/dev/null || true
unzip -o -q "/kaggle/working/song-search/$GEO"
DLL=$(find . -name '*.dll' | head -1)
echo "mod dll: $DLL" | tee -a "$LOG"
python3 - "$DLL" <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import sys, struct
def pe_info(p):
    d = open(p,'rb').read()
    assert d[:2]==b'MZ'
    e = struct.unpack_from('<I', d, 0x3c)[0]
    assert d[e:e+4]==b'PE\x00\x00'
    mach  = struct.unpack_from('<H', d, e+4)[0]
    chars = struct.unpack_from('<H', d, e+22)[0]
    magic = struct.unpack_from('<H', d, e+24)[0]
    return dict(
      machine={0x14c:'i386 (32-bit)',0x8664:'x86_64 (64-bit)',0xaa64:'arm64'}.get(mach,hex(mach)),
      is_dll=bool(chars & 0x2000),
      pe_format='PE32+' if magic==0x20b else 'PE32',
      size=len(d))
i = pe_info(sys.argv[1])
for k,v in i.items(): print('  %s: %s' % (k,v))
ok = i['machine'].startswith('x86_64') and i['is_dll'] and i['pe_format']=='PE32+'
print('  ARCH MATCHES GD 2.2081 (x86_64):', 'YES' if ok else 'NO')
PY

log "8. mod.json sanity"
python3 - <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import json
m = json.load(open('/kaggle/working/song-search/mod.json'))
print('id     :', m['id'])
print('version:', m['version'])
print('geode  :', m['geode'])
print('gd.win :', m['gd']['win'])
assert m['gd']['win'] == '2.2081'
print('mod.json OK')
PY

# --- keep the kernel output small ---
# `kaggle kernels output` downloads everything under /kaggle/working, and the
# build tree holds ~30MB of static libs. Copy the deliverables out and drop
# the build tree so the package can actually be fetched.
cd /kaggle/working/song-search
log "9. pruning build tree / staging output"
if [ -f build/matcool.song-search.geode ]; then
  cp -f build/matcool.song-search.geode /kaggle/working/matcool.song-search.geode
  cp -f build/matcool.song-search.dll   /kaggle/working/matcool.song-search.dll 2>/dev/null || true
fi
cp -f /kaggle/working/BUILD-LOG.txt /kaggle/working/song-search/BUILD-LOG.txt 2>/dev/null || true
rm -rf build/_deps
rm -rf build/bindings
rm -f  build/bindings/*.lib build/*.exp build/*.ilk build/CMakeFiles/*/*.obj 2>/dev/null || true
du -sh /kaggle/working/song-search /kaggle/working/*.geode 2>/dev/null | tee -a "$LOG"
log "ALL DONE"
